# Pandas — Technical Reference

## Quick Index

| Pattern | When to use | Section |
| :--- | :--- | :--- |
| Data inspection | First look at any DataFrame | §1 |
| Selection & filtering | Subset rows and columns | §2 |
| NULL handling | Missing values | §3 |
| Data manipulation | Clean, transform, create columns | §4 |
| Joins & merges | Combine DataFrames | §5 |
| Aggregation & GroupBy | Summarize, reshape, pivot | §6 |
| Window operations | Rank, shift, cumulative, rolling | §7 |
| Date & time | Date arithmetic, extraction | §8 |
| String operations | Text cleaning and extraction | §9 |
| I/O | Read and write data | §10 |
| Performance tips | Speed and memory optimization | §11 |

---
## When to Use

| Signal | Pandas method to reach for |
| :--- | :--- |
| First look at a DataFrame | `.info()`, `.describe()`, `.head()` |
| Filter rows by condition | `.query()` or boolean indexing |
| Select rows and columns by label | `.loc[rows, cols]` |
| Select rows and columns by position | `.iloc[rows, cols]` |
| Replace NULLs | `.fillna()` or `.ffill()` / `.bfill()` |
| Apply condition to create a column | `np.where()` (binary) or `np.select()` (multi) |
| Summarize with aggregation | `.groupby().agg()` |
| Apply aggregation but keep original shape | `.groupby().transform()` |
| Rank within a group | `.groupby().rank(method='dense')` |
| Compare to previous / next row | `.groupby().shift(1)` / `.shift(-1)` |
| Running total within a group | `.groupby().cumsum()` |
| Pivot rows to columns | `.pivot_table()` |
| Combine two DataFrames on a key | `.merge()` |
| Find rows with no match | `merge(..., how='left', indicator=True)` |
| Moving average | `.rolling(n).mean()` |
| Parse date strings | `pd.to_datetime()` |
| Extract date parts | `.dt.year`, `.dt.month`, `.dt.dayofweek` |
| String pattern match | `.str.contains()` |
| Extract regex groups from string | `.str.extract()` |

---
## §1 — Data Inspection

Always run these first. They tell you shape, dtypes, missing values, and distributions before you touch the data.

In [ ]:
import pandas as pd
import numpy as np

# Shape and preview
df.shape                        # (rows, cols)
df.head(5)                      # first 5 rows
df.tail(5)                      # last 5 rows

# Column types and null counts
df.info()                       # dtypes + non-null counts per column
df.dtypes                       # just the dtypes

# Summary statistics
df.describe()                   # numeric columns: count, mean, std, min, quartiles, max
df.describe(include='all')      # include categorical columns too

# Cardinality and distributions
df['col'].nunique()             # number of distinct values
df['col'].value_counts()        # frequency of each value, sorted descending
df['col'].value_counts(normalize=True)  # as proportions (0–1)

# NULL summary across all columns
df.isna().sum()                 # null count per column
df.isna().sum() / len(df)       # null rate per column

**Common mistakes:**
- Skipping `.info()` and assuming dtypes — date columns often load as `object` and need explicit parsing
- Using `.describe()` without checking `.info()` first — `describe()` silently skips non-numeric columns unless `include='all'` is set
- Confusing `df['col'].count()` (excludes NaN) with `len(df)` (includes all rows)

---
## §2 — Selection & Filtering

Three distinct tools: `[]` for quick column access, `.loc` for label-based selection, `.iloc` for position-based selection. Always use `.loc` when selecting rows and columns together.

In [ ]:
# Column selection
df['col']                           # single column → Series
df[['col1', 'col2']]                # multiple columns → DataFrame

# .loc — label-based (rows by index label, cols by name)
df.loc[0]                           # row with index label 0
df.loc[0:5, 'col1':'col3']          # rows 0–5, columns col1 through col3 (inclusive)
df.loc[mask, ['col1', 'col2']]      # filtered rows, specific columns

# .iloc — position-based (integer index only)
df.iloc[0]                          # first row
df.iloc[0:5, 0:3]                   # first 5 rows, first 3 columns (exclusive end)
df.iloc[-1]                         # last row

# Boolean indexing
df[df['age'] > 30]                  # single condition
df[(df['age'] > 30) & (df['region'] == 'US')]   # AND — use & not 'and'
df[(df['age'] < 18) | (df['age'] > 65)]         # OR  — use | not 'or'
df[~(df['status'] == 'inactive')]               # NOT — use ~
df[df['platform'].isin(['iOS', 'Android'])]     # IN
df[~df['user_id'].isin(blocked_ids)]            # NOT IN

# .query() — readable string syntax
df.query("age > 30 and region == 'US'")
df.query("`column name` > 0")                   # backticks for column names with spaces
threshold = 100
df.query("revenue > @threshold")                # @ prefix to reference Python variables

**`.loc` vs `.iloc` vs `[]` comparison:**

| | Rows | Columns | End inclusive? |
| :--- | :--- | :--- | :--- |
| `[]` | ❌ (only with boolean mask) | ✅ by name | n/a |
| `.loc` | ✅ by label | ✅ by name | ✅ Yes |
| `.iloc` | ✅ by position | ✅ by position | ❌ No |

**Common mistakes:**
- Using `and` / `or` / `not` instead of `&` / `|` / `~` in boolean indexing — raises `ValueError`
- Forgetting parentheses around each condition: `df[df['a'] > 1 & df['b'] == 2]` evaluates incorrectly; always wrap each condition
- Chained indexing `df['col'][mask]` instead of `df.loc[mask, 'col']` — triggers `SettingWithCopyWarning` and may not modify the original

---
## §3 — NULL Handling

NULLs affect filtering, aggregation, and joins. Understand the behavior before writing any pipeline.

In [ ]:
# Detect NULLs
df.isna()                                       # IS NULL — element-wise boolean
df.notna()                                      # IS NOT NULL
df['col'].isna().sum()                          # count NULLs in a column

# Fill NULLs
df['salary'].fillna(0)                          # IFNULL(salary, 0)
df['contact'] = df['phone'].fillna(df['email']).fillna('Unknown')  # COALESCE(phone, email, 'Unknown')
df['col'].fillna(df['col'].mean())              # fill with column mean

# Forward / backward fill within a group
df['value'] = df.groupby('user_id')['value'].ffill()  # fill from previous row in group
df['value'] = df.groupby('user_id')['value'].bfill()  # fill from next row in group

# Drop rows with NULLs
df.dropna()                                     # drop rows with ANY null
df.dropna(subset=['user_id', 'date'])           # drop rows where specific columns are null
df.dropna(how='all')                            # drop rows where ALL values are null

# NULL behavior in aggregations
df['col'].sum()     # NaN ignored (treated as 0 in sum)
df['col'].mean()    # NaN ignored — denominator is non-null count, not len(df)
df['col'].count()   # excludes NaN
len(df)             # includes NaN rows

**Common mistakes:**
- Using `df['col'] == None` or `df['col'] == np.nan` — NaN is never equal to anything, including itself; always use `.isna()`
- `fillna(0)` before `.mean()` when NULLs should be excluded — changes the denominator and deflates the mean
- `ffill()` without sorting first — fills in row order, not logical time order; always `sort_values` before forward filling

---
## §4 — Data Manipulation

Covers column creation, renaming, type casting, and conditional logic — the day-to-day cleaning toolkit.

In [ ]:
# Add / overwrite columns
df['new_col'] = df['a'] + df['b']              # vectorized arithmetic
df = df.assign(new_col=df['a'] + df['b'])      # chainable version

# Rename columns
df.rename(columns={'old': 'new', 'a': 'b'})   # rename specific columns
df.columns = ['col1', 'col2', 'col3']          # rename all columns at once

# Drop columns or rows
df.drop(columns=['col1', 'col2'])              # drop columns
df.drop(index=[0, 1, 2])                       # drop rows by index label

# Type casting
df['col'].astype(int)
df['col'].astype(float)
df['col'].astype(str)
df['col'].astype('category')                   # memory-efficient for low-cardinality strings

# Conditional logic
# np.where — binary (IF/ELSE)
df['spender_type'] = np.where(df['spend'] > 100, 'High', 'Low')

# np.select — multi-condition (CASE WHEN equivalent)
conditions = [
    df['spend'] > 100,
    df['spend'].between(50, 100)
]
choices = ['High', 'Medium']
df['spender_type'] = np.select(conditions, choices, default='Low')

# pd.cut — bin continuous values into labeled ranges
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 18, 35, 60, 100],
    labels=['Under 18', '18-35', '36-60', '60+']
)

# map — replace values using a dictionary (SQL CASE WHEN on discrete values)
df['platform_label'] = df['platform_code'].map({1: 'iOS', 2: 'Android', 3: 'Web'})

# apply — row-wise or column-wise custom function (use sparingly — slow)
df['col'] = df['col'].apply(lambda x: x.strip().lower())  # acceptable on strings
df['result'] = df.apply(lambda row: row['a'] + row['b'], axis=1)  # avoid if vectorized op exists

# Deduplication
df.drop_duplicates()                           # drop fully duplicate rows
df.drop_duplicates(subset=['user_id'])         # keep first occurrence per user_id
df.drop_duplicates(subset=['user_id'], keep='last')  # keep last occurrence

**Common mistakes:**
- Conditions in `np.select` are evaluated top to bottom — first matching condition wins; order matters
- `df['col'].map(dict)` returns NaN for keys not in the dictionary — add `.fillna()` if needed
- Using `apply(axis=1)` (row-wise) when a vectorized operation exists — it's a Python loop and 10–100x slower
- `inplace=True` does not speed things up and makes chaining impossible — avoid it

**Choosing a value-substitution / labeling tool:**

| | Selects on | Unmatched values become | SQL analogue |
| :--- | :--- | :--- | :--- |
| `np.where(cond, x, y)` | one boolean condition | n/a (binary) | `IF` / `CASE WHEN ... ELSE` |
| `np.select(conds, choices, default)` | many conditions (first wins) | `default` | multi-branch `CASE WHEN` |
| `s.map(dict)` | discrete value lookup | **NaN** | `CASE` on discrete values |
| `s.replace(dict)` | discrete value lookup | **unchanged** | targeted `REPLACE` |

The `map`-vs-`replace` split is the gotcha: `map` blanks out keys it doesn't know, `replace` leaves them alone.

**`where` vs `mask` (exact opposites):**

| | Keeps original value where | Replaces where |
| :--- | :--- | :--- |
| `s.where(cond, other)` | cond is **True** | cond is False |
| `s.mask(cond, other)` | cond is **False** | cond is True |

**`map` vs `apply` (on a Series):** use `map` for a dict/Series lookup or simple function (faster, clearer); use `apply` only for arbitrary functions a `map` can't express.

**`pd.cut` vs `pd.qcut`:** `cut` makes equal-**width** bins from edges you supply; `qcut` makes equal-**frequency** (quantile) bins. Use `qcut` for balanced bucket sizes, `cut` when the thresholds carry meaning.


In [2]:
import pandas as pd
s = pd.Series([10, 20, 30, 40])
print("where(s>20) keeps where TRUE :", s.where(s>20, 0).tolist())
print("mask(s>20)  keeps where FALSE:", s.mask(s>20, 0).tolist())

vals = pd.Series([1,2,3,4,5,6,7,8,100])
print("\ncut  (equal-width) bucket sizes:", pd.cut(vals, bins=3).value_counts().sort_index().tolist())
print("qcut (equal-freq)  bucket sizes:", pd.qcut(vals, q=3).value_counts().sort_index().tolist())


where(s>20) keeps where TRUE : [0, 0, 30, 40]
mask(s>20)  keeps where FALSE: [10, 20, 0, 0]

cut  (equal-width) bucket sizes: [8, 0, 1]
qcut (equal-freq)  bucket sizes: [3, 3, 3]


---
## §5 — Joins & Merges

`.merge()` is the primary tool — use it for all column-based joins. `.join()` joins on index and is rarely needed.

In [ ]:
# Basic merge — SQL JOIN equivalents
A.merge(B, on='id', how='inner')        # INNER JOIN
A.merge(B, on='id', how='left')         # LEFT JOIN
A.merge(B, on='id', how='right')        # RIGHT JOIN
A.merge(B, on='id', how='outer')        # FULL OUTER JOIN

# Join on multiple keys
A.merge(B, on=['user_id', 'date'], how='left')

# Join on differently named columns
A.merge(B, left_on='user_id', right_on='id', how='inner')

# Handle overlapping column names
A.merge(B, on='id', how='left', suffixes=('_a', '_b'))

# Find rows in A with NO match in B — LEFT JOIN ... WHERE B.id IS NULL
merged = A.merge(B, on='id', how='left', indicator=True)
no_match = merged.loc[merged['_merge'] == 'left_only'].drop(columns='_merge')

# Self join — join a DataFrame to itself
employees.merge(
    employees[['id', 'name']].rename(columns={'id': 'manager_id', 'name': 'manager_name'}),
    on='manager_id',
    how='left'
)

# Stack DataFrames vertically — SQL UNION ALL
pd.concat([df1, df2], ignore_index=True)        # UNION ALL
pd.concat([df1, df2]).drop_duplicates()         # UNION (distinct)

**Common mistakes:**
- Merging on columns with different dtypes (e.g. `int` vs `str`) — merge silently produces no matches; cast first
- Forgetting `suffixes` when both DataFrames have overlapping column names — pandas adds `_x` / `_y` by default
- Using `.join()` instead of `.merge()` — `.join()` joins on index by default; `.merge()` is almost always what you want
- `pd.concat` without `ignore_index=True` — preserves original indices, causing duplicate index values

**`merge` vs `join`, and `concat` vs `merge`:**

| | Aligns on | Default keys | Prefer when |
| :--- | :--- | :--- | :--- |
| `A.merge(B, on='id')` | **columns** | none — you specify `on` | Almost always — explicit, SQL-like, handles renamed keys |
| `A.join(B)` | the **index** | the index | You've set a meaningful index and want to glue several frames on it |

`.join()` silently aligns by index/position, so if your key is an ordinary column it matches the **wrong rows**. Prefer `.merge()` unless you deliberately set an index.

| | Combines by | Use for |
| :--- | :--- | :--- |
| `pd.concat([A, B])` | stacking rows (or columns) — no key matching | `UNION ALL` / gluing partitions |
| `A.merge(B, on=k)` | matching a key | `JOIN` |


In [1]:
import pandas as pd
A = pd.DataFrame({"id":[1,2,3], "val":["a","b","c"]})
B = pd.DataFrame({"id":[2,3,4], "score":[20,30,40]})

print("merge on 'id' (inner) — matches on the key:")
print(A.merge(B, on="id", how="inner"))

print("\njoin — aligns on INDEX, so it pairs the WRONG rows:")
print(A.join(B, lsuffix="_A", rsuffix="_B"))


merge on 'id' (inner) — matches on the key:
   id val  score
0   2   b     20
1   3   c     30

join — aligns on INDEX, so it pairs the WRONG rows:
   id_A val  id_B  score
0     1   a     2     20
1     2   b     3     30
2     3   c     4     40


---
## §6 — Aggregation & GroupBy

Three distinct tools with different output shapes: `agg` collapses rows, `transform` preserves shape, `pivot_table` reshapes across two dimensions.

In [ ]:
# SQL pipeline equivalent
# SELECT col1, AGG(col2) FROM t WHERE cond GROUP BY col1 HAVING ... ORDER BY ... LIMIT n
result = (
    df.query('condition')                           # WHERE
      .groupby('col1', as_index=False)              # GROUP BY
      .agg(agg_col=('col2', 'sum'))                 # SELECT + aggregate
      .query('agg_col > 100')                       # HAVING
      .sort_values('col1', ascending=False)         # ORDER BY
      .head(10)                                     # LIMIT
)

# Basic aggregations
len(df)                                 # COUNT(*)
df['col'].count()                       # COUNT(col) — excludes NULLs
df['user_id'].nunique()                 # COUNT(DISTINCT user_id)
df['sales'].sum()                       # SUM
df['price'].mean()                      # AVG

# Named aggregation in groupby — preferred syntax
result = df.groupby('region', as_index=False).agg(
    total_sales  = ('sales',   'sum'),
    avg_price    = ('price',   'mean'),
    unique_users = ('user_id', 'nunique'),
    min_score    = ('score',   'min')
)

# Conditional aggregation — SUM(CASE WHEN ...) equivalent
df['ios_rev'] = df['revenue'].where(df['platform'] == 'iOS', 0)        # else 0
df['ios_rev_excl'] = df['revenue'].where(df['platform'] == 'iOS')      # else NaN (excluded from mean)
result = df.groupby('region', as_index=False).agg(
    ios_total = ('ios_rev',      'sum'),
    ios_avg   = ('ios_rev_excl', 'mean')
)

# transform — aggregation that keeps original shape (window function equivalent)
df['group_total'] = df.groupby('region')['revenue'].transform('sum')
df['pct_of_group'] = df['revenue'] / df['group_total']

# pivot_table — GROUP BY across two categorical dimensions
df.pivot_table(
    index='region',                     # rows
    columns='platform',                 # columns
    values='revenue',                   # values
    aggfunc='sum',                      # aggregation
    fill_value=0                        # replace NaN with 0
).reset_index()

# crosstab — frequency count only (pivot_table shortcut)
pd.crosstab(df['region'], df['platform'])

**`agg` vs `transform` vs `pivot_table`:**

| | Output shape | SQL equivalent | Use when |
| :--- | :--- | :--- | :--- |
| `agg` | Collapsed (one row per group) | `GROUP BY` | You want a summary table |
| `transform` | Same as input | Window function | You need the result back on every row |
| `pivot_table` | Reshaped (groups become columns) | `GROUP BY` + conditional agg | Two categorical dimensions |

**Common mistakes:**
- Forgetting `as_index=False` in `groupby` — group keys become the index, not columns
- Using `agg` when you need `transform` — `agg` collapses rows; you lose the original index
- `pivot_table` without `fill_value=0` — missing combinations produce NaN instead of 0
- `.agg(['sum', 'mean'])` with a list produces a MultiIndex — use named aggregation syntax to avoid this

**`pivot` vs `pivot_table`, and three "count" methods:**

| | Aggregates duplicates? | Takes `aggfunc`? | Errors on dup keys? |
| :--- | :--- | :--- | :--- |
| `df.pivot(...)` | ❌ reshape only | ❌ | ✅ raises `ValueError` |
| `df.pivot_table(...)` | ✅ | ✅ | ❌ aggregates them |

Use `pivot` only when each index/column pair is already unique; otherwise `pivot_table`.

| Method | Counts | SQL equivalent |
| :--- | :--- | :--- |
| `s.value_counts()` | rows per distinct **value** of one column | `GROUP BY col` + `COUNT(*)` |
| `df.groupby(k).size()` | rows per **group** (includes NaN) | `GROUP BY k` + `COUNT(*)` |
| `s.nunique()` | number of **distinct** values | `COUNT(DISTINCT col)` |

**`drop_duplicates` vs `groupby().first()`:**

| | Keeps | Other columns |
| :--- | :--- | :--- |
| `df.drop_duplicates(subset=k)` | first row per key | kept as-is from that row |
| `df.groupby(k).first()` | first non-null per column per group | can differ column-by-column; lets you aggregate the rest |


---
## §7 — Window Operations

Covers ranking, lag/lead, running totals, and rolling windows — the Pandas equivalents of SQL window functions.

In [ ]:
# Ranking — SQL RANK / DENSE_RANK / ROW_NUMBER equivalents
df['row_num']   = df['score'].rank(method='first',  ascending=False).astype(int)  # ROW_NUMBER
df['rnk']       = df['score'].rank(method='min',    ascending=False).astype(int)  # RANK
df['dense_rnk'] = df['score'].rank(method='dense',  ascending=False).astype(int)  # DENSE_RANK

# Ranking within a group — RANK() OVER (PARTITION BY ...)
df['rank_in_group'] = (
    df.groupby('region')['revenue']
      .rank(method='dense', ascending=False)
      .astype(int)
)

# LAG / LEAD — shift within a group (always sort first)
df = df.sort_values(['user_id', 'date'])
df['prev_value'] = df.groupby('user_id')['value'].shift(1)    # LAG(value, 1)
df['next_value'] = df.groupby('user_id')['value'].shift(-1)   # LEAD(value, 1)

# Running totals — SUM OVER (ROWS UNBOUNDED PRECEDING)
df['run_sum'] = df.groupby('user_id')['amount'].cumsum()
df['run_max'] = df.groupby('user_id')['amount'].cummax()
df['run_min'] = df.groupby('user_id')['amount'].cummin()

# Rolling window — moving average over last n rows
df['moving_avg_7'] = (
    df.groupby('user_id')['amount']
      .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

# Expanding window — cumulative from start of series
df['expanding_mean'] = df.groupby('user_id')['amount'].transform(lambda x: x.expanding().mean())

**Rank method comparison:**

| method | Ties | Gaps after ties | SQL equivalent |
| :--- | :--- | :--- | :--- |
| `first` | No — sequential | n/a | `ROW_NUMBER()` |
| `min` | Yes — share min rank | Yes | `RANK()` |
| `dense` | Yes — share rank | No | `DENSE_RANK()` |

**Common mistakes:**
- Forgetting to `sort_values` before `shift` — shift operates on current row order, not logical order
- Using `groupby().cumsum()` without sorting — produces incorrect running totals
- Using `rolling()` directly on a grouped column without `transform` — loses the group alignment

---
## §8 — Date & Time

Always parse date strings with `pd.to_datetime()` first. All `.dt` accessor operations require a proper datetime dtype.

In [ ]:
# Parse strings to datetime
df['event_ts'] = pd.to_datetime(df['event_ts'])                     # auto-detect format
df['event_ts'] = pd.to_datetime(df['event_ts'], format='%Y-%m-%d')  # explicit format (faster)

# Unix timestamp conversion
df['dt'] = pd.to_datetime(df['ts_col'], unit='s')    # 10-digit (seconds) → datetime
df['dt'] = pd.to_datetime(df['ts_col'], unit='ms')   # 13-digit (milliseconds) → datetime

# Extract components — .dt accessor
df['date']        = df['event_ts'].dt.date                           # DATE()
df['year']        = df['event_ts'].dt.year                           # EXTRACT(YEAR)
df['month']       = df['event_ts'].dt.month                          # EXTRACT(MONTH)
df['day']         = df['event_ts'].dt.day
df['day_of_week'] = df['event_ts'].dt.dayofweek                      # 0=Mon, 6=Sun
df['hour']        = df['event_ts'].dt.hour
df['year_month']  = df['event_ts'].dt.to_period('M').astype(str)     # DATE_FORMAT('%Y-%m')

# Date arithmetic
df['plus_7d']   = df['order_date'] + pd.Timedelta(days=7)            # DATE_ADD(..., INTERVAL 7 DAY)
df['minus_1mo'] = df['order_date'] - pd.DateOffset(months=1)         # DATE_SUB(..., INTERVAL 1 MONTH)
df['days_diff'] = (df['end_date'] - df['start_date']).dt.days        # DATEDIFF

# Current date
today = pd.Timestamp.today().normalize()                              # CURRENT_DATE
df_recent = df.loc[df['event_date'] >= today - pd.Timedelta(days=7)]

# Gaps & Islands — consecutive streak detection
logins = logins.sort_values(['user_id', 'login_date'])
logins['row_num'] = logins.groupby('user_id').cumcount() + 1
logins['grp'] = logins['login_date'] - pd.to_timedelta(logins['row_num'], unit='D')
streaks = (
    logins.groupby(['user_id', 'grp'], as_index=False)
          .agg(streak_start=('login_date', 'min'),
               streak_end  =('login_date', 'max'),
               streak_len  =('login_date', 'count'))
)

**`pd.Timedelta` vs `pd.DateOffset`:**

| | Fixed duration | Calendar-aware | Use for |
| :--- | :--- | :--- | :--- |
| `pd.Timedelta(days=7)` | ✅ | ❌ | Days, hours, minutes |
| `pd.DateOffset(months=1)` | ❌ | ✅ | Months, years (variable length) |

**Common mistakes:**
- Forgetting `pd.to_datetime()` before using `.dt` accessor — raises `AttributeError` if column is still a string
- Using `pd.Timedelta` for months — months have variable lengths; use `pd.DateOffset` instead
- Using `.dt.date` (returns Python `date` objects) when you need `.dt.normalize()` (returns `Timestamp`) — the former breaks further date arithmetic

---
## §9 — String Operations

All string methods live under the `.str` accessor. They are vectorized and NaN-safe — NaN rows return NaN rather than raising an error.

In [ ]:
# Case and whitespace
df['col'].str.lower()
df['col'].str.upper()
df['col'].str.strip()                           # remove leading/trailing whitespace
df['col'].str.strip().str.lower()               # chain operations

# Contains / starts with / ends with
df['col'].str.contains('pattern')               # LIKE '%pattern%'
df['col'].str.contains('pattern', na=False)     # treat NaN as False, not NaN
df['col'].str.startswith('prefix')
df['col'].str.endswith('suffix')

# Replace
df['col'].str.replace('old', 'new')             # REPLACE(col, 'old', 'new')
df['col'].str.replace(r'\d+', '', regex=True)   # remove all digits (regex)

# Split
df['col'].str.split(',')                        # returns list in each cell
df['col'].str.split(',', expand=True)           # expand into separate columns
df[['first', 'last']] = df['name'].str.split(' ', n=1, expand=True)  # split into named cols

# Extract — pull out groups using regex
df['col'].str.extract(r'(\d{4})')               # extract first 4-digit number into a column
df['col'].str.extract(r'(?P<year>\d{4})-(?P<month>\d{2})')  # named groups → named columns

# Length
df['col'].str.len()                             # character count per cell

# Slice
df['col'].str[:3]                               # first 3 characters
df['col'].str[2:5]                              # characters 2–4

**Common mistakes:**
- `str.contains()` without `na=False` — NaN rows return NaN (not False), which breaks boolean indexing
- Using Python `str` methods directly on a Series instead of `.str.*` — raises `AttributeError`
- `str.replace()` without `regex=False` when the pattern has regex special characters (`.`, `+`, `*`) — always set `regex=False` for literal replacements

---
## §10 — I/O

Covers reading and writing data. Setting dtypes on load avoids silent type errors downstream.

In [ ]:
# Read CSV
df = pd.read_csv('file.csv')
df = pd.read_csv('file.csv', usecols=['user_id', 'date', 'revenue'])  # load only needed columns
df = pd.read_csv('file.csv', dtype={'user_id': str, 'amount': float}) # specify dtypes on load
df = pd.read_csv('file.csv', parse_dates=['event_date'])              # parse dates on load
df = pd.read_csv('file.csv', nrows=1000)                              # load first N rows only

# Process large files in chunks
chunks = pd.read_csv('large.csv', chunksize=100_000)
result = pd.concat([chunk[chunk['revenue'] > 0] for chunk in chunks], ignore_index=True)

# Read JSON
df = pd.read_json('file.json')
df = pd.read_json('file.json', lines=True)                            # JSON Lines format

# Read Excel
df = pd.read_excel('file.xlsx', sheet_name='Sheet1')

# Read SQL
import sqlite3
conn = sqlite3.connect('db.sqlite')
df = pd.read_sql('SELECT * FROM table WHERE date > "2024-01-01"', conn)

# Write
df.to_csv('output.csv', index=False)            # index=False avoids writing the row index
df.to_json('output.json', orient='records')
df.to_excel('output.xlsx', index=False, sheet_name='Results')

**Common mistakes:**
- Forgetting `index=False` in `to_csv` — writes the numeric row index as the first column
- Loading all columns when only a few are needed — use `usecols` to reduce memory
- Not specifying `dtype` for ID columns — numeric IDs load as `int64` and lose leading zeros (e.g. zip codes)

---
## §11 — Performance Tips

In [ ]:
# Prefer vectorized operations over apply
df['col'] * 2                           # fast — vectorized C loop
df['col'].apply(lambda x: x * 2)        # slow — row-by-row Python loop

# .query() vs boolean indexing
df.query("age > 30 and region == 'US'") # readable, slightly faster on large DataFrames
df[(df['age'] > 30) & (df['region'] == 'US')]  # equivalent, better for dynamic conditions

# Reduce memory with category dtype (low-cardinality string columns)
df['platform'] = df['platform'].astype('category')

# Read only needed columns
pd.read_csv('file.csv', usecols=['user_id', 'date', 'revenue'])

# Avoid chained indexing — use .loc instead
df.loc[mask, 'col'] = value             # correct
df[mask]['col'] = value                 # wrong — may not modify original (SettingWithCopyWarning)

# Use .copy() when slicing a DataFrame you intend to modify
subset = df[df['region'] == 'US'].copy()
subset['new_col'] = 1                   # safe — modifies copy, not original

# Process large files in chunks
chunks = pd.read_csv('large.csv', chunksize=100_000)
result = pd.concat([process(chunk) for chunk in chunks])

- Avoid `apply` with `axis=1` (row-wise) — it is a Python loop; use `np.where`, `np.select`, or vectorized column ops instead
- `inplace=True` does not speed things up and prevents chaining — avoid it
- For string columns with few unique values (platform, country, status), `.astype('category')` can reduce memory by 50–90%
- `df.eval('a + b')` can be faster than `df['a'] + df['b']` for large DataFrames — uses `numexpr` under the hood

---
# Decision Guide

```
Aggregating data?
├── One metric, one grouping dimension        → groupby().agg()
├── One metric across two categorical dims    → pivot_table()
├── Frequency count only                      → crosstab()
└── Need result back on every original row    → groupby().transform()

Filtering rows?
├── Static, readable condition                → .query()
└── Dynamic / programmatic condition          → boolean indexing df[mask]

Selecting rows + columns together?
├── By label                                  → .loc[rows, cols]
└── By position                               → .iloc[rows, cols]

Labeling / bucketing values?
├── Binary condition                           → np.where(cond, true_val, false_val)
├── Multiple conditions                        → np.select(conditions, choices, default)
└── Binning continuous values                  → pd.cut() or pd.qcut()

Combining DataFrames?
├── Join on a column                           → .merge(on='col')
├── Rows with no match in other table          → merge(..., how='left', indicator=True)
└── Stack rows vertically                      → pd.concat([df1, df2])

Window operations?
├── Rank within group                          → groupby().rank(method='dense')
├── Previous / next row value                  → groupby().shift(1) / shift(-1)
├── Running total                              → groupby().cumsum()
└── Moving average over last n rows            → groupby().transform(lambda x: x.rolling(n).mean())

NULL handling?
├── Replace with a value                       → .fillna(value)
├── Replace with previous row in group         → groupby().ffill()
└── Drop rows                                  → .dropna(subset=[...])

Speeding up slow code?
├── Using apply row-wise                       → replace with np.where / vectorized ops
├── Large file loading slowly                  → usecols= or chunksize=
└── High-cardinality string columns            → .astype('category')
```